# Example 7.7 / 1: Model Zero or a Minimum with a Binary

Companion notebook for **7.7 Semi-Continuous Variables and Special Ordered Sets** in the AMPL lab Documentation, section "Model Zero or a Minimum with a Binary". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Model Zero or a Minimum with a Binary

With `0 < L <= U`, a binary `Use` and two rows give exactly "zero, or between `L` and `U`":

$$
L \cdot \text{Use} \le x \le U \cdot \text{Use}.
$$

When `Use = 0` both rows force `x = 0`; when `Use = 1` they keep `x` in `[L, U]`. In the steel
model, each product that is made at all must be made in at least `least[p]` tons:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                         # products
param rate {PROD} > 0;            # tons produced per hour
param profit {PROD};              # profit per ton
param market {PROD} >= 0;         # most that can be sold, in tons
param least {p in PROD} >= 0, <= market[p];  # minimum lot when the product is made
param avail >= 0;                 # hours available

var Make {p in PROD} >= 0, <= market[p];  # tons produced
var Use {PROD} binary;                     # 1 if the product is made

maximize TotalProfit: sum {p in PROD} profit[p] * Make[p];
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;  # hours available
subject to LowerWhenUsed {p in PROD}: Make[p] >= least[p] * Use[p];   # at least the lot when on
subject to UpperWhenUsed {p in PROD}: Make[p] <= market[p] * Use[p];  # zero when off

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["least"] = {"bands": 1000, "coils": 1500, "plate": 2000}
ampl.param["avail"] = 40

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.get_data("Make", "Use").to_pandas())

HiGHS 1.14.0: optimal solution; objective 195500
6 simplex iterations
1 branching nodes
       Make  Use
bands  5500    1
coils     0    0
plate  2000    1


### Pure AMPL: data.dat and model.run

Without minimum lots the best plan is 6,000 tons of bands and 1,600 of plate, which earns
196,400. Plate now needs at least 2,000 tons, so the model gives up 500 tons of bands to make room
for the full lot of plate, and coils stay at exactly zero.

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param: PROD: rate profit market least :=
  bands  200  25  6000  1000
  coils  140  30  4000  1500
  plate  160  29  3500  2000 ;
param avail := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Make, Use, TotalProfit;

HiGHS 1.14.0: optimal solution; objective 195500
6 simplex iterations
1 branching nodes
:       Make Use    :=
bands   5500   1
coils      0   0
plate   2000   1
;

TotalProfit = 195500



### Another way in amplpy

A lower bound alone removes the off state. Fixing every `Use[p]` to 1 has the same effect as
declaring `Make[p] >= least[p]`; with the same `model.mod` and `data.py`:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["least"] = {"bands": 1000, "coils": 1500, "plate": 2000}
ampl.param["avail"] = 40

In [11]:
# run.py
for p in ["bands", "coils", "plate"]:
    ampl.var["Use"][p].fix(1)
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result, ampl.obj["TotalProfit"].value())
print(ampl.var["Make"].to_dict())

solved 186928.57142857142
{'bands': 3357.142857142857, 'coils': 1500, 'plate': 2000}
